In [5]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_train_clean3.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_val_clean3.parquet')

print(train.shape, val.shape)

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

(616293, 35) (133316, 35)


In [6]:
# ============================================================
# 1. 라이브러리 불러오기
# ============================================================

import numpy as np
import optuna

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

c:\Users\soohok402\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ============================================================
# 2. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [8]:
# ============================================================
# 2. 종속변수 설정
# ============================================================

y_train = train[target_col]
y_val = val[target_col]

In [9]:
# ============================================================
# 3. station_id 원핫인코딩 독립변수 생성
# ============================================================

# OHE 전 독립변수 데이터셋
X_train_fe = train[num_cols_fe]
X_val_fe = val[num_cols_fe]

# 원핫인코딩
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 X
    sparse_output=True  # sparse: 희소한, 희소행렬(0이 아닌 값이 드문 행렬)
    # 원핫인코딩 결과를 전체 다 저장 X, 희소한 1의 값의 위치만 저장(메모리 사용 적음)
)

# 학습은 train만
ohe.fit(train[['station_id']])  # sklearn 인코더는 2차원 행렬을 받는다.

# 변환
train_st_id_ohe = ohe.transform(train[['station_id']])
val_st_id_ohe = ohe.transform(val[['station_id']])
# 여기까지는 일반적은 인코딩 과정

# train, val 셋에 합치기
from scipy.sparse import hstack
# scipy안에 희소행렬을 다루는 서브캐키지 scipy.sparse
# hstack(horizontal stack)은 concat(axis=1)과 같은 역할, df가 아닌 행렬도 붙일 수 있는 함수

X_train_with_station_fe = hstack([
    X_train_fe,
    train_st_id_ohe
]).tocsr()  # df + 희소행력(ohe) -> csr형태로 저장하여 sparse 상태 유지
# 이 변수는 더이상 df가 아님

X_val_with_station_fe = hstack([
    X_val_fe,
    val_st_id_ohe
]).tocsr()

print("원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 들어남")
print(X_train_fe.shape, X_train_with_station_fe.shape)
print(X_val_fe.shape, X_val_with_station_fe.shape)
# 제공된 데이터셋의 관측소 개수는 465개(md파일에 명시)
# 기존컬럼 31개 + 465 = 496이 정상

원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 들어남
(616293, 31) (616293, 496)
(133316, 31) (133316, 496)


In [14]:
# ============================================================
# 7. 모델15 (최종검증: 기존 catboost모델, y_val)
# ============================================================

# 모델 정의
model_base_iter_3000 = CatBoostRegressor(
    iterations=3000,
    random_state=42
)

# 모델 학습
model_base_iter_3000.fit(
    X_train_with_station_fe,
    y_train,
    eval_set=(X_val_with_station_fe, y_val),
    early_stopping_rounds=100,
    verbose=200
)

pred_base_iter_3000 = model_base_iter_3000.predict(X_val_with_station_fe)
pred_train_base_iter_3000 = model_base_iter_3000.predict(X_train_with_station_fe)

rmse_val_base_iter_3000 = root_mean_squared_error(
    y_val,
    pred_base_iter_3000
)

rmse_train_base_iter_3000 = root_mean_squared_error(
    y_train,
    pred_train_base_iter_3000
)

print('val RMSE: ', rmse_val_base_iter_3000)
print('train RMSE: ', rmse_train_base_iter_3000)
print(
    'Gap: ',
    rmse_val_base_iter_3000 - rmse_train_base_iter_3000
)


Learning rate set to 0.071425
0:	learn: 0.3896247	test: 0.3445404	best: 0.3445404 (0)	total: 25.2ms	remaining: 1m 15s
200:	learn: 0.2319619	test: 0.2046232	best: 0.2046159 (199)	total: 4.76s	remaining: 1m 6s
400:	learn: 0.2204046	test: 0.2000967	best: 0.2000654 (397)	total: 9.74s	remaining: 1m 3s
600:	learn: 0.2131298	test: 0.1979693	best: 0.1979693 (600)	total: 14.2s	remaining: 56.9s
800:	learn: 0.2077914	test: 0.1967202	best: 0.1967199 (799)	total: 18.9s	remaining: 51.9s
1000:	learn: 0.2032345	test: 0.1959183	best: 0.1959183 (1000)	total: 23.8s	remaining: 47.6s
1200:	learn: 0.1994702	test: 0.1953818	best: 0.1953818 (1200)	total: 28.9s	remaining: 43.2s
1400:	learn: 0.1959837	test: 0.1949824	best: 0.1949814 (1399)	total: 33.4s	remaining: 38.1s
1600:	learn: 0.1929611	test: 0.1946357	best: 0.1946256 (1595)	total: 37.7s	remaining: 33s
1800:	learn: 0.1900792	test: 0.1943698	best: 0.1943251 (1757)	total: 42.5s	remaining: 28.3s
2000:	learn: 0.1874984	test: 0.1940757	best: 0.1940757 (2000)	to

In [15]:
# ============================================================
# 7. 모델15 (최종검증: 기존 catboost모델, y_val)
# ============================================================

# 모델 정의
model_base = CatBoostRegressor(
    random_state=42
)

# 모델 학습
model_base.fit(
    X_train_with_station_fe,
    y_train,
)

pred_base = model_base.predict(X_val_with_station_fe)
pred_train_base = model_base.predict(X_train_with_station_fe)

rmse_val_base = root_mean_squared_error(
    y_val,
    pred_base
)

rmse_train_base = root_mean_squared_error(
    y_train,
    pred_train_base
)

print('val RMSE: ', rmse_val_base)
print('train RMSE: ', rmse_train_base)
print(
    'Gap: ',
    rmse_val_base - rmse_train_base
)


Learning rate set to 0.112971
0:	learn: 0.3821263	total: 23.3ms	remaining: 23.3s
1:	learn: 0.3646604	total: 44.7ms	remaining: 22.3s
2:	learn: 0.3497715	total: 67.2ms	remaining: 22.3s
3:	learn: 0.3371794	total: 90.9ms	remaining: 22.6s
4:	learn: 0.3262559	total: 113ms	remaining: 22.5s
5:	learn: 0.3171331	total: 134ms	remaining: 22.2s
6:	learn: 0.3090710	total: 158ms	remaining: 22.4s
7:	learn: 0.3023203	total: 180ms	remaining: 22.4s
8:	learn: 0.2963395	total: 204ms	remaining: 22.5s
9:	learn: 0.2912331	total: 225ms	remaining: 22.2s
10:	learn: 0.2867859	total: 249ms	remaining: 22.3s
11:	learn: 0.2832142	total: 268ms	remaining: 22.1s
12:	learn: 0.2800538	total: 289ms	remaining: 22s
13:	learn: 0.2771835	total: 309ms	remaining: 21.8s
14:	learn: 0.2748794	total: 329ms	remaining: 21.6s
15:	learn: 0.2725816	total: 350ms	remaining: 21.5s
16:	learn: 0.2707189	total: 370ms	remaining: 21.4s
17:	learn: 0.2690486	total: 392ms	remaining: 21.4s
18:	learn: 0.2673156	total: 414ms	remaining: 21.4s
19:	learn